In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import DPOTrainer, DPOConfig
import torch

from peft import LoraConfig, get_peft_model, PeftModel

import mlflow

/workspace/repos/dragon-llm/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data

## Loading

In [2]:
cache_dir = "/workspace/hf_cache"
dataset_name = "ewin-reg/Stock-Market-Trading-Signals"  # "argilla/llama-2-banking-preference"
banking_pref_ds = load_dataset(dataset_name, cache_dir=cache_dir)

## Formatting

In [3]:
def hard_negative_verdict(correct_verdict: str) -> str:
    "See suggestion here: https://huggingface.co/datasets/ewin-reg/Stock-Market-Trading-Signals"
    match correct_verdict:
        case "BUY":
            return "HOLD"
        case "SELL":
            return "HOLD"
        case "HOLD":
            return "BUY"
        case _:
            raise ValueError(f"Unknown verdict: {correct_verdict}")

def format_prompt(system_prompt: dict[str, str], user_prompt: dict[str, str]) -> str:
    return {
        "role": user_prompt["role"],
        "content": f"{system_prompt['content']}\n{user_prompt['content']}",
    }

def extract_verdict(answer: str) -> str:
    return answer.split("VERDICT: ")[-1]

def format_data_dpo(sample: dict[str, list[dict[str, str]]]) -> dict[str, str]:
    messages = sample["messages"]

    # Create hard negative
    correct_answer = messages[2]
    correct_verdict = extract_verdict(correct_answer['content'])
    wrong_verdict = hard_negative_verdict(correct_verdict)
    wrong_answer = correct_answer.copy()
    wrong_answer['content'] = wrong_answer['content'].replace(f"VERDICT: {correct_verdict}", f"VERDICT: {wrong_verdict}")

    return {
        "prompt": messages[:2], # format_prompt(system_prompt, user_prompt),
        "chosen": [correct_answer],
        "rejected": [wrong_answer],
        # "correct_verdict": correct_verdict,
        # "wrong_verdict": wrong_verdict,
    }


In [4]:
dpo_ds = banking_pref_ds.map(format_data_dpo, remove_columns=["messages"])
dpo_ds["train"][0]

{'prompt': [{'role': 'system',
   'content': 'You are a financial analyst specializing in stock direction prediction. You analyze technical indicators and historical prices to predict stock direction, generating step-by-step reasoning chains followed by a final VERDICT: BUY, SELL, or HOLD.'},
  {'role': 'user',
   'content': 'Analyze AAPL as of 2018-08-31 and predict the stock direction.\n\nPrice history (last 20 days): $49.01 -> $48.80 -> $49.22 -> $49.39 -> $49.71 -> $51.45 -> $50.84 -> $50.95 -> $51.53 -> $52.72 -> $53.82 -> $53.82 (current)\nCurrent price: $53.82\n\nTechnical indicators:\n- RSI(14): 89.3\n- Volume Multiplier: 1.62x\n- Volatility Multiplier: 0.93x\n\nBased on this data, should an investor BUY, SELL, or HOLD AAPL? Answer with a step-by-step reasoning chain and a final VERDICT.'}],
 'chosen': [{'role': 'assistant',
   'content': '<think>\n1. Momentum: RSI is 89.3, indicating an overbought (> 68) state.\n2. Liquidity: Volume Multiplier is 1.62x, showing elevated volume

# Model
## Loading

In [5]:
model_name = "OVHaiLLM/Llama-Open-Finance-8B"
cache_dir = "/workspace/hf_cache"
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=cache_dir
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=cache_dir,
    device_map="auto",  # cpu
    dtype="auto",
)
model

Loading weights: 100%|██████████| 291/291 [00:21<00:00, 13.26it/s]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128009)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4

In [6]:
# original model
sample = dpo_ds["validation"][0]
prompt_text = tokenizer.apply_chat_template(
    sample["prompt"],
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

model.eval()
with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

prediction = tokenizer.decode(
    output_ids[0, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print("Prediction:\n", prediction)
print("\nReference:\n", sample["chosen"][0]["content"])

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Prediction:
 **Step-by-Step Reasoning Chain:**

1. **Current Price and Recent Trend**: The stock has recently traded at $88.90, with a slight upward trend over the past few days. The price has been fluctuating between $88.11 and $90.23, indicating a range-bound movement.

2. **RSI(14) Analysis**: The Relative Strength Index (RSI) is at 55.1, which is in the **neutrally bullish range** (typically between 30 and 70). This suggests the stock is not overbought or oversold, indicating a balanced market sentiment. A reading of 55.1 indicates the stock is neither overextended to the upside nor downside, suggesting a potential for further movement in either direction.

3. **Volume Multiplier**: The Volume Multiplier is 0.69x, which is **below 1**. This indicates that the volume of trades is lower than the average volume over the past 20 days. Lower volume suggests less market participation, which could indicate a lack of strong conviction in the current price direction. This could be a warning

## PEFT with LoRA

In [7]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model, lora_config)
peft_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128009)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_p

In [8]:
print("Total number of parameters: ", round(peft_model.get_model_status().total_params / 1e9, 2), "G")
print("Number of trainable parameters: ", round(peft_model.get_model_status().trainable_params / 1e6, 2), "M")

Total number of parameters:  8.03 G
Number of trainable parameters:  3.41 M


# DPO training

In [9]:
dpo_config = DPOConfig(
    num_train_epochs=2,
    # max_steps=100,  # for debugging
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    dataloader_num_workers=16,
    learning_rate=1e-6,
    optim="adamw_torch_fused",
    tf32=True,
    torch_compile=False,
    report_to="mlflow",
    run_name="dpo-trading-signals",
    logging_strategy="steps",
    logging_steps=10,
    eval_strategy="no",  # do evaluation at beginning and end only
    eval_on_start=True,
    save_strategy="no",
    # load_best_model_at_end=True,
    loss_type=["sigmoid", "bco_pair", "sft"],  # loss types to combine
    loss_weights=[0.8, 0.2, 1.0]  # corresponding weights, as used in the MPO paper
)

In [10]:
if "ref" in peft_model.peft_config:
    # DPOTrainer adds a frozen reference adapter to PEFT models. Remove one left
    # behind by a previous trainer initialization (for example, after interrupting training).
    peft_model.delete_adapter("ref")

dpo_trainer = DPOTrainer(
    model=peft_model,
    train_dataset=dpo_ds["train"],
    eval_dataset=dpo_ds["validation"], # .select(range(100)),
    processing_class=tokenizer,
    args=dpo_config,
)

Dropping fully truncated examples from train dataset: 100%|██████████| 43122/43122 [00:01<00:00, 39849.24 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 7611/7611 [00:00<00:00, 38044.33 examples/s]


In [ ]:
mlflow.set_system_metrics_sampling_interval(120)

with mlflow.start_run(
    run_name="dpo-trading-signals",
    log_system_metrics=True,
):
    dpo_trainer.train()  # initial evaluation already done (eval_on_start=True)
    dpo_trainer.evaluate()
    # model_info = mlflow.transformers.log_model(
    #     transformers_model={
    #         "model": peft_model,
    #         "tokenizer": tokenizer,
    #     },
    #     name="tuned_model",
    # )


2026/10/25 22:15:47 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/10/25 22:15:47 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0,No log,2.554028,1.169026,0.000000,-1.306907,-1.310863,0.652594,0.000000,0.000000,0.000000,0.000000,-201.612343,-206.447967
10,2.538180,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log
20,2.546219,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log
30,2.566951,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log
40,2.526867,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log
50,2.549396,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log,No Log


In [ ]:
save_dir = "/workspace/models/finance-dpo-lora"

peft_model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

('/workspace/models/finance-dpo-lora/tokenizer_config.json',
 '/workspace/models/finance-dpo-lora/chat_template.jinja',
 '/workspace/models/finance-dpo-lora/tokenizer.json')

## Comparison

In [ ]:

save_dir = "/workspace/models/finance-dpo-lora"

# base_model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     cache_dir=cache_dir,
#     device_map="auto",
#     dtype="auto",
# )
tuned_model = PeftModel.from_pretrained(model, save_dir)
# tokenizer = AutoTokenizer.from_pretrained(save_dir)

/workspace/repos/dragon-llm/.venv/lib/python3.14/site-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
# tuned_model
sample = dpo_ds["validation"][0]
prompt_text = tokenizer.apply_chat_template(
    sample["prompt"],
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

tuned_model.eval()
with torch.inference_mode():
    output_ids = tuned_model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

prediction = tokenizer.decode(
    output_ids[0, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print("Prediction:\n", prediction)
print("\nReference:\n", sample["chosen"][0]["content"])

Prediction:
 **Step-by-Step Reasoning Chain:**

1. **Current Price and Recent Trend**: The stock has recently traded at $88.90, with a slight upward trend over the past few days. The price has been fluctuating between $88.11 and $90.23, indicating a range-bound movement.

2. **RSI(14) Analysis**: The Relative Strength Index (RSI) is at 55.1, which is in the **neutrally bullish range** (typically between 30 and 70). This suggests the stock is not overbought or oversold, indicating a balanced market sentiment. A reading of 55.1 indicates the stock is neither overextended to the upside nor downside, suggesting a potential for further movement in either direction.

3. **Volume Multiplier**: The Volume Multiplier is 0.69x, which is **below 1**. This indicates that the volume of trades is lower than the average volume over the past 20 days. Lower volume suggests less market participation, which could indicate a lack of strong conviction in the current price direction. This could be a warning